# 📦 Analyse du Chiffre d'Affaires — Projet E-Commerce
> 

Ce notebook réalise l'analyse complète des ventes : calcul du CA Brut, CA Net, TVA, identification du meilleur produit et export des résultats.

**Données utilisées :** `ventes_clean.csv`

## ⚙️ Étape 1 — Importation des bibliothèques 

In [1]:
import pandas as pd
import numpy as np

# Chargement des données
df = pd.read_csv('ventes_clean.csv')

print(f'✅ Données chargées : {len(df)} ventes, {len(df.columns)} colonnes')
print(f'   Colonnes : {list(df.columns)}')
print('\n📊 Aperçu des données :')
df.head(10)

✅ Données chargées : 98 ventes, 4 colonnes
   Colonnes : ['ID', 'Prix', 'Quantite', 'Remise']

📊 Aperçu des données :


,ID,Prix,Quantite,Remise
0,100,9.88,4,0.0
1,101,178.98,5,0.0
2,102,50.37,9,30.0
3,103,86.81,5,0.0
4,104,36.13,5,0.0
5,105,79.09,6,0.0
6,106,109.56,7,0.0
7,107,127.58,6,0.0
8,108,119.95,5,5.0
9,109,79.12,14,10.0


## 📦 Étape 2 — Calcul du Chiffre d'Affaires Brut
`CA_Brut = Prix × Quantité`

In [ ]:
df['CA_Brut'] = (df['Prix'] * df['Quantite']).round(2)
ca_brut_total = df['CA_Brut'].sum()

print(f'💰 CA Brut Total : {ca_brut_total:,.2f} €')
print('\n📊 Détail par transaction :')
df[['ID', 'Prix', 'Quantite', 'CA_Brut']].head(10)

### 📝 Interprétation — CA Brut
Le **CA Brut** représente le chiffre d'affaires théorique si aucune remise n'était accordée aux clients. C'est le **plafond maximal** de revenus que l'entreprise pourrait atteindre.

## 💸 Étape 3 — Calcul du CA Net (après remises)
`CA_Net = CA_Brut × (1 - Remise%)`

In [ ]:
df['CA_Net'] = (df['CA_Brut'] * (1 - df['Remise'] / 100)).round(2)
ca_net_total = df['CA_Net'].sum()
remise_totale = ca_brut_total - ca_net_total
taux_remise = (remise_totale / ca_brut_total) * 100

print(f'💸 Remises accordées : {remise_totale:,.2f} € ({taux_remise:.1f}% du CA Brut)')
print(f'📈 CA Net Total      : {ca_net_total:,.2f} €')
print('\n📊 Comparaison CA Brut vs CA Net :')
df[['ID', 'Prix', 'Quantite', 'Remise', 'CA_Brut', 'CA_Net']].head(10)

### 📝 Interprétation — CA Net
Le **CA Net** est le revenu réellement encaissé après déduction des remises commerciales. C'est l'indicateur de performance le plus fiable pour évaluer la santé financière réelle de l'entreprise.

## 🏦 Étape 4 — Calcul de la TVA (20%)
`TVA = CA_Net × 0.20`

In [ ]:
df['TVA_20pct'] = (df['CA_Net'] * 0.20).round(2)
tva_totale = df['TVA_20pct'].sum()

print(f'🏦 TVA Totale collectée : {tva_totale:,.2f} €')
print('\n📊 Aperçu TVA par transaction :')
df[['ID', 'CA_Net', 'TVA_20pct']].head(10)

### 📝 Interprétation — TVA
La TVA collectée est perçue par l'entreprise **pour le compte de l'État**. Ce montant **ne constitue pas un revenu** pour l'entreprise.

## ✅ Étape 5 — CA Total de l'entreprise & Récapitulatif

In [ ]:
ca_ttc_total = ca_net_total + tva_totale

resume = pd.DataFrame({
    'Indicateur': [
        'CA Brut Total',
        'Remises accordées',
        'CA Net Total (HT)',
        'TVA collectée (20%)',
        'CA TTC Total'
    ],
    'Montant (€)': [
        ca_brut_total,
        -remise_totale,
        ca_net_total,
        tva_totale,
        ca_ttc_total
    ]
})

print('=' * 50)
print('     📊 RÉCAPITULATIF FINANCIER 📊')
print('=' * 50)
for _, row in resume.iterrows():
    print(f'{row["Indicateur"]:25} : {row["Montant (€)"]:,.2f} €')
print('=' * 50)

## 🏆 Étape 6 — Meilleure vente (plus gros CA Net)

In [ ]:
idx_max = df['CA_Net'].idxmax()
best_sale = df.loc[idx_max]
pct_contribution = (best_sale['CA_Net'] / ca_net_total) * 100

print(f'🏆 Meilleure transaction (ID {best_sale["ID"]:.0f}) :')
print(f'   Prix unitaire    : {best_sale["Prix"]:.2f} €')
print(f'   Quantité         : {best_sale["Quantite"]}')
print(f'   Remise           : {best_sale["Remise"]:.0f}%')
print(f'   CA Net généré    : {best_sale["CA_Net"]:,.2f} €')
print(f'   Contribution     : {pct_contribution:.2f}% du CA Net total')

## 📊 Étape 7 — Statistiques générales des ventes

In [ ]:
print('=' * 50)
print('     📊 STATISTIQUES DES VENTES')
print('=' * 50)

print(f'\n📈 Nombre total de transactions : {len(df)}')
print(f'📊 Quantité totale vendue      : {df["Quantite"].sum():,.0f} unités')
print(f'💰 Prix moyen unitaire         : {df["Prix"].mean():,.2f} €')
print(f'📦 Quantité moyenne par vente  : {df["Quantite"].mean():.1f} unités')
print(f'🎯 Remise moyenne appliquée    : {df["Remise"].mean():.1f}%')
print(f'🔝 CA Net maximum par vente    : {df["CA_Net"].max():,.2f} €')
print(f'⚠️  CA Net minimum par vente    : {df["CA_Net"].min():,.2f} €')

## 📈 Étape 8 — Distribution des remises

In [ ]:
remise_counts = df['Remise'].value_counts().sort_index()

print('\n📊 Distribution des remises :')
print('-' * 50)
for remise, count in remise_counts.items():
    ca_net_remise = df[df['Remise'] == remise]['CA_Net'].sum()
    pct_ventes = (count / len(df)) * 100
    print(f'   Remise {remise:5.0f}% : {count:3} ventes ({pct_ventes:5.1f}%) - CA Net: {ca_net_remise:12,.2f} €')

## 💾 Étape 9 — Export des résultats

In [ ]:
df.to_csv('resultats_final_.csv', index=False, encoding='utf-8-sig')

print('✅ Fichier exporté avec succès : resultats_final_.csv')

## 🏁 Conclusion générale

In [ ]:
print('\n' + '=' * 55)
print('                 🏁 CONCLUSION GÉNÉRALE 🏁')
print('=' * 55)
print(f'''
┌─────────────────────────────────────────────────────────────┐
│                    SYNTHÈSE FINANCIÈRE                      │
├─────────────────────────────────────────────────────────────┤
│  CA Brut Total      : {ca_brut_total:>27,.2f} €  │
│  Remises accordées  : {remise_totale:>27,.2f} €  │
│  CA Net Total (HT)  : {ca_net_total:>27,.2f} €  │
│  TVA collectée (20%) : {tva_totale:>27,.2f} €  │
│  CA TTC Total       : {ca_ttc_total:>27,.2f} €  │
├─────────────────────────────────────────────────────────────┤
│  Meilleure vente    : ID {best_sale["ID"]:.0f} → {best_sale["CA_Net"]:>13,.2f} € │
│  Contribution       : {pct_contribution:>27.2f}%  │
│  Nombre transactions: {len(df):>27}     │
│  Quantité totale    : {df["Quantite"].sum():>27,.0f} u   │
└─────────────────────────────────────────────────────────────┘
''')

print("\n✨ Analyse terminée avec succès !")